# Policy attributes

This notebook documents steps from raw IEA data to the final
`policies.json` file used by the frontend.

**Pipeline:**
1. Load & clean IEA PAMS export
2. OpenAI GPT-4o-mini classification (instrument type, legally binding, quantified target)
3. Rule-based sector mapping (topic fields + keyword fallback)
4. Export JSON file

In [14]:
pip install pandas openai plotly nbformat ipykernel beautifulsoup4


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [15]:
import pandas as pd
import json
import os
import re
import time
import numpy as np
from openai import OpenAI
from bs4 import BeautifulSoup

In [16]:
df = pd.read_csv("IEA_PAMS_Export 2_21_2026, 7_04_19 PM.csv")
print(f"{df.shape[0]} rows, {df.shape[1]} columns")
df.head(2)

12470 rows, 16 columns


,title,description,status,year,jurisdiction,source,datePromulgated,yearEnded,learnMore,learnMoreLanguage,dateModified,countries,states,technologies,tags,policyType
0,Law 47-09 on Energy Efficiency,"<div>Law 47-09, or the ""Law on Energy Efficien...",In force,2011,National,International Energy Agency,29 Septembre 2011,NaN,http://www.muat.gov.ma/sites/default/files/Reg...,NaN,1636982975000,"[{""iso3"":""MAR"",""name"":""Morocco""}]",[],[],[],[]
1,Smart Farming Initiative,The Smart Farming programme focuses on eight k...,In force,2016,National,International Energy Agency,NaN,NaN,https://smartfarming.ie/,English,1618996947000,"[{""iso3"":""IRL"",""name"":""Ireland""}]",[],[],[],[]


In [17]:
# ── Clean HTML tags from descriptions ──────────────────────────
def clean_html(text):
    if pd.isna(text):
        return ""
    return BeautifulSoup(text, "html.parser").get_text(separator=" ").strip()

df['description_clean'] = df['description'].apply(clean_html)

# ── Parse policyType JSON into flat columns ────────────────────
def parse_policy_type(pt_str):
    try:
        pts = json.loads(pt_str)
        if not pts:
            return pd.Series({'topic': None, 'family': None, 'category': None})
        topics = [p.get('topic') for p in pts if p.get('topic')]
        families = [p.get('family') for p in pts if p.get('family')]
        categories = [p.get('category') for p in pts if p.get('category')]
        return pd.Series({
            'topic': ' | '.join(set(topics)) if topics else None,
            'family': ' | '.join(set(families)) if families else None,
            'category': ' | '.join(set(categories)) if categories else None,
        })
    except:
        return pd.Series({'topic': None, 'family': None, 'category': None})

df[['topic', 'family', 'category']] = df['policyType'].apply(parse_policy_type)

# ── Quick check ────────────────────────────────────────────────
labeled = df['topic'].notna().sum()
print(f"Labeled: {labeled} ({labeled/len(df)*100:.1f}%) | Unlabeled: {len(df)-labeled} ({(len(df)-labeled)/len(df)*100:.1f}%)")
print(f"\nSample cleaned description:\n{df['description_clean'].iloc[0][:200]}")
print(f"\nTop topics:\n{df['topic'].value_counts().head(8)}")

Labeled: 6542 (52.5%) | Unlabeled: 5928 (47.5%)

Sample cleaned description:
Law 47-09, or the "Law on Energy Efficiency" aims to increase energy efficiency in the use of energy sources, to avoid waste, to reduce the energy costs on the national economy, and to enhance sustain

Top topics:
topic
Fuels                1295
Economy-wide          715
Methane abatement     612
Critical Minerals     597
Power                 567
Transport             550
Buildings             546
Just transitions      355
Name: count, dtype: int64


## Step 2 — OpenAI GPT-4o-mini Classification

We used GPT-4o-mini to classify each policy's `instrument_type`,
`legally_binding` status, and whether it `has_quantified_target`.

The cells below show the prompting code (commented out to avoid
re-running of API calls). Results are saved in
`classifications_progress.json` and loaded in the next section.

In [18]:
# ── OpenAI client setup + single-row test ──────────────────────
# (Commented out — already run; results saved to disk)

"""
client = OpenAI(api_key="api key")

test_title = df['title'].iloc[0]
test_desc = df['description_clean'].iloc[0][:1000]

response = client.chat.completions.create(
    model="gpt-4o-mini",
    temperature=0,
    response_format={"type": "json_object"},
    messages=[{"role": "user", "content": f\"\"\"Classify this climate policy. Return ONLY valid JSON with these fields:

- instrument_type: one of [carbon_tax, cap_and_trade, subsidy, tax_credit,
  feed_in_tariff, mandate, ban, standard, voluntary_agreement, labeling,
  reporting, framework_legislation, other]
- sector: one of [energy, transport, buildings, industry, agriculture,
  economy_wide, other]
- legally_binding: true or false
- stringency: integer 1-5 (1=aspirational/voluntary, 5=strict mandate with penalties)
- has_quantified_target: true or false

Title: {test_title}
Description: {test_desc}\"\"\"
    }]
)

result = json.loads(response.choices[0].message.content)
print(f"Policy: {test_title}")
print(f"Result: {json.dumps(result, indent=2)}")
"""

'\nclient = OpenAI(api_key="api key")\n\ntest_title = df[\'title\'].iloc[0]\ntest_desc = df[\'description_clean\'].iloc[0][:1000]\n\nresponse = client.chat.completions.create(\n    model="gpt-4o-mini",\n    temperature=0,\n    response_format={"type": "json_object"},\n    messages=[{"role": "user", "content": f"""Classify this climate policy. Return ONLY valid JSON with these fields:\n\n- instrument_type: one of [carbon_tax, cap_and_trade, subsidy, tax_credit,\n  feed_in_tariff, mandate, ban, standard, voluntary_agreement, labeling,\n  reporting, framework_legislation, other]\n- sector: one of [energy, transport, buildings, industry, agriculture,\n  economy_wide, other]\n- legally_binding: true or false\n- stringency: integer 1-5 (1=aspirational/voluntary, 5=strict mandate with penalties)\n- has_quantified_target: true or false\n\nTitle: {test_title}\nDescription: {test_desc}"""\n    }]\n)\n\nresult = json.loads(response.choices[0].message.content)\nprint(f"Policy: {test_title}")\nprin

In [19]:
# ── Classification function + 10-row test ──────────────────────
# (Commented out — already run)

"""
def classify_policy(title, description):
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            temperature=0,
            response_format={"type": "json_object"},
            messages=[{"role": "user", "content": f\"\"\"Classify this climate policy. Return ONLY valid JSON with these fields:

- instrument_type: one of [carbon_tax, cap_and_trade, subsidy, tax_credit,
  feed_in_tariff, mandate, ban, standard, voluntary_agreement, labeling,
  reporting, framework_legislation, other]
- legally_binding: true or false
- has_quantified_target: true or false

Title: {title}
Description: {description[:1000]}\"\"\"
            }]
        )
        return json.loads(response.choices[0].message.content)
    except Exception as e:
        return {"error": str(e)}

# Test on 10 spread-out rows
test_indices = [0, 50, 200, 500, 1000, 2000, 4000, 6000, 8000, 10000]
for i in test_indices:
    result = classify_policy(df['title'].iloc[i], df['description_clean'].iloc[i])
    print(f"\nRow {i}: {df['title'].iloc[i][:70]}")
    print(f"  → {result}")
    time.sleep(0.5)
"""

'\ndef classify_policy(title, description):\n    try:\n        response = client.chat.completions.create(\n            model="gpt-4o-mini",\n            temperature=0,\n            response_format={"type": "json_object"},\n            messages=[{"role": "user", "content": f"""Classify this climate policy. Return ONLY valid JSON with these fields:\n\n- instrument_type: one of [carbon_tax, cap_and_trade, subsidy, tax_credit,\n  feed_in_tariff, mandate, ban, standard, voluntary_agreement, labeling,\n  reporting, framework_legislation, other]\n- legally_binding: true or false\n- has_quantified_target: true or false\n\nTitle: {title}\nDescription: {description[:1000]}"""\n            }]\n        )\n        return json.loads(response.choices[0].message.content)\n    except Exception as e:\n        return {"error": str(e)}\n\n# Test on 10 spread-out rows\ntest_indices = [0, 50, 200, 500, 1000, 2000, 4000, 6000, 8000, 10000]\nfor i in test_indices:\n    result = classify_policy(df[\'title\'].i

In [20]:
# ── Batch classification with multithreading ───────────────────
# (Commented out — already run; ~12,000 rows classified)

"""
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

SAVE_PATH = "classifications_progress.json"

if os.path.exists(SAVE_PATH):
    with open(SAVE_PATH, 'r') as f:
        classifications = json.load(f)
    print(f"Resuming from {len(classifications)} already classified rows")
else:
    classifications = {}

def classify_row(idx):
    return str(idx), classify_policy(
        df['title'].iloc[idx],
        df['description_clean'].iloc[idx]
    )

remaining = [i for i in range(len(df)) if str(i) not in classifications]
print(f"{len(remaining)} rows remaining")

try:
    with ThreadPoolExecutor(max_workers=5) as executor:
        futures = {executor.submit(classify_row, idx): idx for idx in remaining}
        for future in tqdm(as_completed(futures), total=len(remaining), desc="Classifying"):
            str_idx, result = future.result()
            classifications[str_idx] = result

            if len(classifications) % 100 == 0:
                with open(SAVE_PATH, 'w') as f:
                    json.dump(classifications, f)

except KeyboardInterrupt:
    print(f"\nInterrupted! Saving {len(classifications)} classifications...")

with open(SAVE_PATH, 'w') as f:
    json.dump(classifications, f)
print(f"\nDone! {len(classifications)} / {len(df)} classified.")
"""

'\nfrom tqdm import tqdm\nfrom concurrent.futures import ThreadPoolExecutor, as_completed\n\nSAVE_PATH = "classifications_progress.json"\n\nif os.path.exists(SAVE_PATH):\n    with open(SAVE_PATH, \'r\') as f:\n        classifications = json.load(f)\n    print(f"Resuming from {len(classifications)} already classified rows")\nelse:\n    classifications = {}\n\ndef classify_row(idx):\n    return str(idx), classify_policy(\n        df[\'title\'].iloc[idx],\n        df[\'description_clean\'].iloc[idx]\n    )\n\nremaining = [i for i in range(len(df)) if str(i) not in classifications]\nprint(f"{len(remaining)} rows remaining")\n\ntry:\n    with ThreadPoolExecutor(max_workers=5) as executor:\n        futures = {executor.submit(classify_row, idx): idx for idx in remaining}\n        for future in tqdm(as_completed(futures), total=len(remaining), desc="Classifying"):\n            str_idx, result = future.result()\n            classifications[str_idx] = result\n\n            if len(classifications

In [21]:
# ── Load saved OpenAI classifications ──────────────────────────
SAVE_PATH = "classifications_progress.json"

with open(SAVE_PATH, 'r') as f:
    classifications = json.load(f)

print(f"Classified: {len(classifications)} / {len(df)} ({len(classifications)/len(df)*100:.1f}%)")

# Merge into dataframe
clf_df = pd.DataFrame.from_dict(classifications, orient='index')
clf_df.index = clf_df.index.astype(int)

for col in ['instrument_type', 'legally_binding', 'has_quantified_target']:
    df[col] = clf_df[col]

# Check coverage
for col in ['instrument_type', 'legally_binding', 'has_quantified_target']:
    n = df[col].notna().sum()
    print(f"{col}: {n} classified ({n/len(df)*100:.1f}%)")

print(f"\n{df['instrument_type'].value_counts()}")

Classified: 10687 / 12470 (85.7%)
instrument_type: 10401 classified (83.4%)
legally_binding: 10401 classified (83.4%)
has_quantified_target: 10401 classified (83.4%)

instrument_type
subsidy                  3738
framework_legislation    1771
standard                 1507
other                    1289
mandate                   923
voluntary_agreement       264
feed_in_tariff            209
tax_credit                174
labeling                  131
ban                       109
cap_and_trade              88
reporting                  83
carbon_tax                 80
tariff                     18
tax                        11
guideline                   2
regulation                  2
auction                     1
quota                       1
Name: count, dtype: int64


In [22]:
# Map off-label values to the correct categories
instrument_cleanup = {
    'tariff': 'feed_in_tariff',
    'tax': 'carbon_tax',
    'guideline': 'voluntary_agreement',
    'regulation': 'standard',
    'auction': 'market_based',
    'quota': 'cap_and_trade',
}
df['instrument_type'] = df['instrument_type'].replace(instrument_cleanup)

print(df['instrument_type'].value_counts())
print(f"\nlegally_binding:\n{df['legally_binding'].value_counts()}")
print(f"\nhas_quantified_target:\n{df['has_quantified_target'].value_counts()}")

instrument_type
subsidy                  3738
framework_legislation    1771
standard                 1509
other                    1289
mandate                   923
voluntary_agreement       266
feed_in_tariff            227
tax_credit                174
labeling                  131
ban                       109
carbon_tax                 91
cap_and_trade              89
reporting                  83
market_based                1
Name: count, dtype: int64

legally_binding:
legally_binding
True     6100
False    4301
Name: count, dtype: int64

has_quantified_target:
has_quantified_target
False    7133
True     3268
Name: count, dtype: int64


## Step 3 — Rule-Based Sector Mapping

Maps each policy to one of 7 frontend sectors using IEA's
existing `policyType` topic/family fields, with a keyword fallback
for the ~47.5% of rows that have no topic assigned.

| Code | Sector |
|------|--------|
| RE   | Renewable Energy Incentives |
| FPD  | Fossil Fuel Phase-Down |
| CPM  | Carbon Pricing and Markets |
| EEF  | Energy Efficiency |
| GRT  | Grid and Transport Decarbonisation |
| LU   | Land Use, Forests and Agriculture |
| CF   | Climate Finance and Governance |

In [23]:
# ── Rule-based sector mapping (no OpenAI needed) ───────────────
TOPIC_SECTOR = {
    'Power':                                  'RE',
    'Technology R&D and innovation':          'RE',
    'Fuels':                                  'FPD',
    'Methane abatement':                      'FPD',
    'Transport':                              'GRT',
    'Critical Minerals':                      'GRT',
    'Buildings':                              'EEF',
    'Industry':                               'EEF',
    'Economy-wide':                           'CF',
    'People-Centred Clean Energy Transitions':'CF',
    'Just transitions':                       'CF',
}
CPM_FAMILIES = {'Economic policies', 'Market pull'}

def _is_null(v):
    return v is None or (isinstance(v, float) and np.isnan(v))

def map_topic_to_sector(topic_str, family_str):
    if _is_null(topic_str):
        return None
    topic = str(topic_str).split(' | ')[0].strip()
    sector = TOPIC_SECTOR.get(topic)
    if topic == 'Economy-wide' and not _is_null(family_str):
        if set(str(family_str).split(' | ')) & CPM_FAMILIES:
            sector = 'CPM'
    return sector

KEYWORD_RULES = [
    ('CPM', r'carbon\s*tax|carbon\s*pric|emission\s*trad|cap.and.trade|\bets\b|carbon\s*market|carbon\s*credit|carbon\s*levy'),
    ('RE',  r'solar|wind\s*(energy|power)|renewable|feed.in\s*tariff|photovoltaic|geothermal|hydropower|bioenergy|biomass|clean\s*energy\s*incentive|green\s*energy'),
    ('FPD', r'coal\s*phase|fossil\s*fuel|methane\s*abat|oil\s*phase|gas\s*phase|fuel\s*subsidy\s*reform|phase.down|phase.out\s*fossil'),
    ('GRT', r'electric\s*vehicle|\bev\b|public\s*transport|rail\s*decarbon|grid\s*modern|transmission\s*grid|charging\s*infra|critical\s*mineral|zero.emission\s*vehicle|low.emission\s*vehicle'),
    ('EEF', r'energy\s*efficiency|insulation|building\s*standard|building\s*code|appliance\s*standard|energy\s*audit|ecodesign|energy\s*performance|minimum\s*energy'),
    ('LU',  r'\bforest\b|land\s*use|agricultur|deforestation|\bredd\b|afforestation|reforestation|wetland|biodiversity|soil\s*carbon|agroforest'),
    ('CF',  r'climate\s*financ|climate\s*fund|\bgovernance\b|net\s*zero|carbon\s*neutral|climate\s*strateg|green\s*bond|just\s*transition|\bndcs?\b|nationally\s*determined'),
]

def keyword_sector(title, description):
    text = (str(title) + ' ' + str(description)).lower()
    for sector, pattern in KEYWORD_RULES:
        if re.search(pattern, text):
            return sector
    return 'CF'  # default to governance/framework

def get_sector(row):
    s = map_topic_to_sector(row.get('topic'), row.get('family'))
    if s:
        return s
    return keyword_sector(row.get('title', ''), row.get('description_clean', ''))

df['sector'] = df.apply(get_sector, axis=1)

from_topic = df['topic'].notna().sum()
print(f"Sector from IEA topic:       {from_topic} ({from_topic/len(df)*100:.1f}%)")
print(f"Sector from keyword fallback: {df['topic'].isna().sum()} ({df['topic'].isna().mean()*100:.1f}%)")
print(f"\nSector distribution:")
print(df['sector'].value_counts().to_string())

Sector from IEA topic:       6542 (52.5%)
Sector from keyword fallback: 5928 (47.5%)

Sector distribution:
sector
RE     3289
CF     2956
EEF    2160
FPD    2056
GRT    1867
CPM      72
LU       70


## Step 4 — Export JSON Files for Frontend

Generate two JSON files:
- **`policies.json`** — one record per policy
- **`country_summary.json`** — per-country rollup with policy counts by sector

In [24]:
# ── Export JSON files for frontend ─────────────────────────────

def safe(v):
    """Convert numpy scalars to JSON-safe Python types."""
    if v is None:
        return None
    if isinstance(v, float) and np.isnan(v):
        return None
    if isinstance(v, (np.integer,)):
        return int(v)
    if isinstance(v, (np.floating,)):
        return round(float(v), 6)
    if isinstance(v, (np.bool_,)):
        return bool(v)
    return v

SCOPE_MAP = {
    'National':                  'national',
    'International':             'international',
    'State/Provincial':          'subnational',
    'City/Municipal':            'local',
    'Regional':                  'regional',
    'International Energy Agency': 'international',
}

# ── 1. policies.json ──────────────────────────────────────────
policies_out = []
for idx, row in df.iterrows():
    try:
        country_list = [{'iso3': c['iso3'], 'name': c['name']}
                        for c in json.loads(row['countries'])]
    except Exception:
        country_list = []

    policies_out.append({
        'id':                   int(idx),
        'title':                row['title'],
        'countries':            country_list,
        'year':                 safe(row['year']),
        'status':               row['status'] if pd.notna(row.get('status')) else None,
        'scope':                SCOPE_MAP.get(row.get('jurisdiction'), row.get('jurisdiction'))
                                if pd.notna(row.get('jurisdiction')) else None,
        'sector':               row['sector'],
        'instrument_type':      safe(row.get('instrument_type')),
        'legally_binding':      safe(row.get('legally_binding')),
        'has_quantified_target':safe(row.get('has_quantified_target')),
    })

with open('policies.json', 'w') as f:
    json.dump(policies_out, f, indent=2)
print(f"✓ policies.json         — {len(policies_out):,} policies")

# ── 2. country_summary.json ───────────────────────────────────
# Explode df so each row = one policy-country pair
exp = df.copy()
exp['country_parsed'] = exp['countries'].apply(
    lambda s: [(c['iso3'], c['name']) for c in json.loads(s)] if pd.notna(s) else []
)
exp = exp.explode('country_parsed')
exp = exp[exp['country_parsed'].apply(lambda x: isinstance(x, tuple))]
exp['iso3']         = exp['country_parsed'].apply(lambda x: x[0])
exp['country_name'] = exp['country_parsed'].apply(lambda x: x[1])

country_summary = []
for iso3, grp in exp.groupby('iso3'):
    sector_counts = grp['sector'].value_counts().to_dict()
    active_count  = int((grp['status'] == 'In force').sum())

    country_summary.append({
        'country':            grp['country_name'].iloc[0],
        'iso3':               iso3,
        'total_policies':     int(len(grp)),
        'active_policies':    active_count,
        'policies_by_sector': {k: int(v) for k, v in sector_counts.items()},
    })

country_summary.sort(key=lambda x: x['total_policies'], reverse=True)

with open('country_summary.json', 'w') as f:
    json.dump(country_summary, f, indent=2)
print(f"✓ country_summary.json  — {len(country_summary)} countries")

✓ policies.json         — 12,470 policies
✓ country_summary.json  — 217 countries
